# DSNML Retail Sales Forecasting

This notebook benchmarks various tree-based regressors on the DSNML dataset, including CatBoost, LightGBM, and XGBoost. It is designed for use in Google Colab or a local Jupyter environment.

## Goals
- load and validate the retail dataset
- engineer deterministic features
- compare model performance with cross-validation
- save a Kaggle-style submission file


In [ ]:
# Install dependencies
!pip install -q pandas numpy matplotlib seaborn scikit-learn catboost lightgbm xgboost

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor
import lightgbm as lgb
import xgboost as xgb

# Optional: mount Google Drive in Colab
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/DSNML')


In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print('Train shape:', train.shape)
print('Test shape:', test.shape)
display(train.head())

In [ ]:
def prepare_data(df):
    df = df.copy()
    for col in ['product_category', 'fat_content']:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown').astype(str).str.lower().str.strip()
    if 'product_weight_kg' in df.columns:
        df['product_weight_kg'] = df['product_weight_kg'].fillna(-1)
        df['weight_clean'] = df['product_weight_kg'].replace(-1, np.nan)
        if 'product_price' in df.columns:
            df['price_per_kg'] = df['product_price'] / df['weight_clean']
    if 'store_size' in df.columns:
        df['store_size'] = df['store_size'].fillna('Unknown').astype(str)
    if {'store_location_tier', 'store_format'}.issubset(df.columns):
        df['store_tier_format'] = df['store_location_tier'].astype(str) + '_' + df['store_format'].astype(str)
    if 'product_category' in df.columns:
        non_edible = ['household', 'health and hygiene', 'hard drinks', 'others']
        df['fat_content_adjusted'] = np.where(df['product_category'].isin(non_edible), 'Non-Edible', df['fat_content'].fillna('Unknown'))
    return df

train = prepare_data(train)
test = prepare_data(test)
display(train.head())

In [ ]:
cat_price = train.groupby('product_category')['product_price'].mean()
store_price = train.groupby('store_code')['product_price'].mean()
store_format_sales = train.groupby('store_format')['total_sales'].mean()
product_sales_mean = train.groupby('product_code')['total_sales'].mean()
product_price_mean = train.groupby('product_code')['product_price'].mean()
global_mean_price = train['product_price'].mean()
global_target_mean = train['total_sales'].mean()

for df in [train, test]:
    df['price_to_cat_mean'] = df['product_price'] / df['product_category'].map(cat_price).fillna(global_mean_price)
    df['price_to_store_mean'] = df['product_price'] / df['store_code'].map(store_price).fillna(global_mean_price)
    df['store_format_sales_mean'] = df['store_format'].map(store_format_sales).fillna(global_target_mean)
    df['product_sales_mean'] = df['product_code'].map(product_sales_mean).fillna(global_target_mean)
    df['product_price_mean'] = df['product_code'].map(product_price_mean).fillna(global_mean_price)
    df['price_ratio_to_product_mean'] = df['product_price'] / df['product_code'].map(product_price_mean).fillna(global_mean_price)
    if 'product_weight_kg' in df.columns:
        df['is_weight_missing'] = df['product_weight_kg'].isna().astype(int)
        df['product_weight_kg'] = df['product_weight_kg'].fillna(-1)

print('Feature engineering complete.')

In [ ]:
TARGET = 'total_sales'
ID_COL = 'id'
cat_features = [
    'product_code', 'fat_content', 'product_category', 'store_code',
    'store_size', 'store_location_tier', 'store_format', 'store_tier_format',
    'fat_content_adjusted'
]

for col in cat_features:
    if col in train.columns:
        train[col] = train[col].fillna('Unknown').astype(str)
    if col in test.columns:
        test[col] = test[col].fillna('Unknown').astype(str)

X = train.drop(columns=[ID_COL, TARGET])
y = train[TARGET]
X_test = test.drop(columns=[ID_COL])
cat_idx = [X.columns.get_loc(c) for c in cat_features if c in X.columns]
print('Feature count:', X.shape[1])
print('Categorical columns:', cat_idx)

In [ ]:
def evaluate_catboost(X, y):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr_idx, va_idx in kf.split(X):
        model = CatBoostRegressor(iterations=1000, depth=6, learning_rate=0.05, l2_leaf_reg=3, random_strength=1, cat_features=cat_idx, random_seed=42, verbose=False)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=(X.iloc[va_idx], y.iloc[va_idx]), early_stopping_rounds=50)
        pred = np.clip(model.predict(X.iloc[va_idx]), 0, None)
        scores.append(root_mean_squared_error(y.iloc[va_idx], pred))
    return np.mean(scores), np.std(scores)

def evaluate_lightgbm(X, y):
    X_eval = X.copy()
    for col in cat_features:
        if col in X_eval.columns:
            X_eval[col] = X_eval[col].astype('category')
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr_idx, va_idx in kf.split(X_eval):
        model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42, objective='regression', metric='rmse', verbosity=-1)
        model.fit(X_eval.iloc[tr_idx], y.iloc[tr_idx], eval_set=[(X_eval.iloc[va_idx], y.iloc[va_idx])], callbacks=[lgb.early_stopping(50, verbose=False)])
        pred = np.clip(model.predict(X_eval.iloc[va_idx]), 0, None)
        scores.append(root_mean_squared_error(y.iloc[va_idx], pred))
    return np.mean(scores), np.std(scores)

cb_mean, cb_std = evaluate_catboost(X, y)
lgb_mean, lgb_std = evaluate_lightgbm(X, y)
print(f'CatBoost RMSE: {cb_mean:.4f} +/- {cb_std:.4f}')
print(f'LightGBM RMSE: {lgb_mean:.4f} +/- {lgb_std:.4f}')

In [ ]:
sns.set_theme(style='whitegrid')
plt.figure(figsize=(10, 6))
train['residual'] = train['total_sales'] - train['total_sales'].mean()
sns.histplot(train['total_sales'], bins=40, kde=True)
plt.title('Target Distribution')
plt.show()

sns.scatterplot(data=train, x='product_price', y='total_sales', alpha=0.5)
plt.title('Price vs Sales')
plt.show()

In [ ]:
final_model = CatBoostRegressor(iterations=1000, depth=6, learning_rate=0.05, l2_leaf_reg=3, random_strength=1, cat_features=cat_idx, random_seed=42, verbose=100)
final_model.fit(X, y)
test_predictions = np.clip(final_model.predict(X_test), 0, None)
submission = pd.DataFrame({'id': test['id'], 'total_sales': test_predictions})
submission.to_csv('final_catboost_submission.csv', index=False)
display(submission.head())